获取statement的分布。


In [3]:
import os
import json
import csv
from glob import glob

In [4]:
def find_json_files(directory):
    return glob(os.path.join(directory, "*.json"))
def load_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

results = []
for json_file in find_json_files("mock object"):
    mo_Result = ["","",""]
    data = load_json(json_file)
    for mo in data:
        if mo.get("mockPatternLevel") == 0:
            mo_Result[0] = "Local"
        else:
            mo_Result[0] = "Shared"
        
        stubtypes = set()
        verification = set()
        for smt in mo.get("statements",[]):
            # 修复 locate 为空的问题
            # fix the issue of empty locate
            smt_locate = smt.get("locate","")
            if smt_locate == "":
                location_context = smt.get("locationContext", {}) or {}
                method_name = location_context.get("methodName", "") or ""
                method_annotations = location_context.get("methodAnnotations", []) or []
                smt_context = smt.get("locationContext",{})
                if "Test" in method_annotations or "test" in method_name.lower():
                    smt_locate= "Test Case"
                
            if smt.get("type") == "STUBBING":
                stubtypes.add(smt_locate)
            elif smt.get("type") == "VERIFICATION":
                verification.add(smt_locate)

        if len(stubtypes) == 0:
            mo_Result[1] = "None"
        elif "Test Case" in stubtypes:
            if len(stubtypes) == 1:
                mo_Result[1] = "Local only"
            else:
                mo_Result[1] = "Partially shared"
        else:
            mo_Result[1] = "Fully shared"

        if len(verification) == 0:
            mo_Result[2] = "None"       
        elif "Test Case" in verification:
            if len(verification) == 1:
                mo_Result[2] = "Local only"
            else:
                mo_Result[2] = "Partially shared"
        else:
            mo_Result[2] = "Fully shared"
        results.append(mo_Result)


In [7]:
import pandas as pd

df = pd.DataFrame(results, columns=["creation", "Stubbing", "Verification"])
df["sum"] = 1
df.to_excel("mock lifecycle.xlsx", index=False)